# 🚦 CanSign-AI — Training Notebook

**Before running:**
1. Go to `Runtime → Change runtime type → T4 GPU` → Save
2. Run Cell 1 to check GPU
3. Run Cell 2 to upload your project files
4. Then run the remaining cells top to bottom
5. After training, download `traffic_sign_model.keras` from the Files panel on the left

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU detected: {gpus[0].name}')
else:
    print('⚠️  No GPU found — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install scikit-learn opencv-python-headless --quiet
print('✅ Dependencies installed')

In [ ]:
# ── Cell 3: Upload project files from your machine ────────────────────────────
#
# This uploads cnn_model.py, data_preprocessing.py, and canadian_labels.py
# directly from your computer into this Colab session.
# After this cell runs, they live at /content/ and can be imported normally.
#
from google.colab import files

print('📂 Upload these 3 files from your local project folder:')
print('   • cnn_model.py')
print('   • data_preprocessing.py')
print('   • canadian_labels.py')
print()

uploaded = files.upload()

required = {'cnn_model.py', 'data_preprocessing.py', 'canadian_labels.py'}
missing  = required - set(uploaded.keys())
if missing:
    raise RuntimeError(f'❌ Missing files: {missing}. Please re-run this cell and upload all three.')

print('✅ All files uploaded successfully')

In [ ]:
# ── Cell 4: Imports (from your actual project files) ──────────────────────────
#
# We import directly from your uploaded files — no copy-pasting.
# If you update cnn_model.py or data_preprocessing.py locally and re-upload,
# training will automatically use your latest code.
#
import numpy as np
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

from cnn_model          import create_cnn_model
from data_preprocessing import load_data, get_augmented_generator, IMG_SIZE, NUM_CLASSES
from canadian_labels    import CANADIAN_SIGNS

print('✅ Imports done — using your project files as the single source of truth')

In [ ]:
# ── Cell 5: Download and extract GTSRB dataset ────────────────────────────────
import os, zipfile, csv, cv2
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

print('⬇️  Downloading GTSRB dataset...')
!wget -q --show-progress https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Training_Images.zip -O train.zip
!wget -q --show-progress https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_Images.zip  -O test.zip
!wget -q --show-progress https://sid.erda.dk/public/archives/daaeac0d7ce1152aea9b61d9f1e19370/GTSRB_Final_Test_GT.zip        -O test_gt.zip
print('✅ Downloads complete')

print('📦 Extracting...')
for z in ['train.zip', 'test.zip', 'test_gt.zip']:
    with zipfile.ZipFile(z, 'r') as zf:
        zf.extractall('.')
print('✅ Extraction complete')

# ── Load training images via data_preprocessing.load_data() ──────────────────
print('🔄 Loading training images via data_preprocessing.load_data()...')
X_train_raw, y_train_raw = load_data('GTSRB/Final_Training/Images')

# ── Load test images using the ground-truth CSV ───────────────────────────────
print('🔄 Loading test images...')
test_images, test_labels = [], []
test_root = 'GTSRB/Final_Test/Images'
gt_path   = 'GT-final_test.csv'

with open(gt_path, newline='') as f:
    reader = csv.DictReader(f, delimiter=';')
    for row in reader:
        img = cv2.imread(os.path.join(test_root, row['Filename']))
        if img is None:
            continue
        img = cv2.resize(img, IMG_SIZE)
        test_images.append(img)
        test_labels.append(int(row['ClassId']))

X_test_raw = np.array(test_images, dtype='float32') / 255.0
y_test_raw = np.array(test_labels)

# ── Combine and split ─────────────────────────────────────────────────────────
X_all = np.concatenate([X_train_raw, X_test_raw])
y_all = np.concatenate([y_train_raw, y_test_raw])

X_train, X_test, y_train_raw_split, y_test_raw_split = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

y_train = to_categorical(y_train_raw_split, num_classes=NUM_CLASSES)
y_test  = to_categorical(y_test_raw_split,  num_classes=NUM_CLASSES)

print(f'✅ Train: {len(X_train)} | Test: {len(X_test)} | Classes: {NUM_CLASSES}')

# ── Free up disk space ────────────────────────────────────────────────────────
!rm train.zip test.zip test_gt.zip
print('🧹 Cleaned up zip files')

In [ ]:
# ── Cell 6: Build model via cnn_model.create_cnn_model() ─────────────────────
#
# Uses your cnn_model.py directly — any architecture changes you make locally
# will be reflected here after re-uploading the file.
#
model = create_cnn_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=NUM_CLASSES)
model.summary()

In [ ]:
# ── Cell 7: Class weights + tf.data pipeline ──────────────────────────────────
y_train_labels    = y_train.argmax(axis=1)
class_weights     = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_labels),
    y=y_train_labels
)
class_weight_dict = dict(enumerate(class_weights))
print(f'📊 Class weights computed for {len(class_weight_dict)} classes')

# Augmentation layers (must be instantiated outside @tf.function)
aug_rotation    = tf.keras.layers.RandomRotation(10/360, seed=42)
aug_zoom        = tf.keras.layers.RandomZoom(0.15, seed=42)
aug_translation = tf.keras.layers.RandomTranslation(0.1, 0.1, seed=42)

def augment(image, label):
    image = aug_rotation(image,    training=True)
    image = aug_zoom(image,        training=True)
    image = aug_translation(image, training=True)
    image = tf.image.random_brightness(image, max_delta=0.2)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=len(X_train), seed=42)
    .map(augment, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .repeat()
    .prefetch(AUTOTUNE)
)

val_ds = (
    tf.data.Dataset.from_tensor_slices((X_test, y_test))
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

steps_per_epoch = len(X_train) // BATCH_SIZE
print(f'✅ tf.data pipeline ready — {steps_per_epoch} steps/epoch')

In [ ]:
# ── Cell 8: Train ─────────────────────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        'traffic_sign_model.keras',
        save_best_only=True,
        monitor='val_accuracy',
        mode='max',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
]

print('🚀 Training on GPU...')
history = model.fit(
    train_ds,
    steps_per_epoch=steps_per_epoch,
    epochs=50,
    validation_data=val_ds,
    class_weight=class_weight_dict,
    callbacks=callbacks,
)

best_val_acc = max(history.history['val_accuracy'])
print(f'\n✅ Training complete')
print(f'🏆 Best validation accuracy: {best_val_acc:.4f}')

In [ ]:
# ── Cell 9: Plot training curves ──────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_title('Accuracy')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True)

ax2.plot(history.history['loss'],     label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_title('Loss')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()
print('📈 Saved training_curves.png')

In [ ]:
# ── Cell 10: Download model ────────────────────────────────────────────────────
from google.colab import files

print('⬇️  Downloading model to your computer...')
files.download('traffic_sign_model.keras')
files.download('training_curves.png')
print('✅ Done — place traffic_sign_model.keras in your project root folder')